In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [4]:
from sklearn.metrics import f1_score
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from pathlib import Path
from argparse import ArgumentParser

import os
import ray
import time
import torch
import json
import torchmetrics
import numpy as np

import torchmetrics.aggregation
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset

from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN
from cluster_intrep_repo.stacks_utils import *
from tqdm.auto import tqdm, trange
from more_itertools import chunked

In [5]:
compute_dtype = torch.bfloat16
device = 'cuda'
model_id = "Qwen/QwQ-32B"

In [6]:

tokenizer = initialize_tokenizer(model_id)

n_blocks = 6


In [7]:
dataset = load_dataset(
    f"dmitriihook/qwq-32b-planning-6-blocks")["train"]

In [9]:
def make_data_to_process(dataset, labels_dict, n_rows, eval_results, answer_type, tokenizer):
    data_to_process = []
    for idx, row in enumerate(dataset.select(range(n_rows))):
        generation = row["generation"]

        if "[PLAN]\n" not in generation:
            continue

        actions = extract_actions(row)
        if actions is None:
            continue

        parsed_actions = parse_block_actions(actions)
        plan_start = generation.index("[PLAN]\n") + len("[PLAN]\n")
        text = generation[:plan_start]
        text = ""

        # need to remove the eos token
        pos_start = len(tokenize_blocksworld_generation(tokenizer, row, text)[0][:-1])
        
        data_to_process.append({
            "idx": idx,
            "pos_start": pos_start,
            "actions": parsed_actions
        })


    return data_to_process

In [10]:
training_data = make_data_to_process(dataset, None, 5000, None, "all", tokenizer)

In [18]:
from collections import Counter

Counter([len(x["actions"][0][1]) for x in training_data if len(x["actions"]) > 0])

Counter({2: 3695, 0: 296, 1: 99})

In [20]:
n_train = int(len(training_data) * 0.9)

train_items = training_data[:n_train]
test_items = training_data[n_train:]

In [42]:
def process_data(items, n_blocks, tgt_action, max_tokens = 3200):
    new_items = []

    for item in items:
        actions = item["actions"]

        try:
            blocks = actions[tgt_action][1]
            block = blocks[0]
            label = block
        except Exception as e:
            continue

        new_items.append({
            "pos_start": item["pos_start"],
            "label": label,
            "idx":  item["idx"],
            "label": label
        })

    return new_items
    

In [43]:
items = process_data(test_items, n_blocks, 0)

In [44]:
Counter([x["label"] for x in items])

Counter({'B': 84, 'A': 75, 'F': 60, 'D': 60, 'C': 54, 'E': 51, 'X': 2})

In [51]:
rows = range(5, 10)

for i, r in enumerate(rows):
    print(
        f"Reasoning trace: {i}"
    )
    print(
        tokenizer.decode(
            tokenize_blocksworld_generation(tokenizer, dataset[r])[0]
        ).split("</think")[0]
    )

Reasoning trace: 0
<|im_start|>user
I am playing with a set of blocks where I need to arrange the blocks into stacks. Here are the actions I can do

Pick up a block
Unstack a block from on top of another block
Put down a block
Stack a block on top of another block

I have the following restrictions on my actions:
I can only pick up or unstack one block at a time.
I can only pick up or unstack a block if my hand is empty.
I can only pick up a block if the block is on the table and the block is clear. A block is clear if the block has no other blocks on top of it and if the block is not picked up.
I can only unstack a block from on top of another block if the block I am unstacking was really on top of the other block.
I can only unstack a block from on top of another block if the block I am unstacking is clear.
Once I pick up or unstack a block, I am holding the block.
I can only put down a block that I am holding.
I can only stack a block on top of another block if I am holding the bloc